## EVTOL & Passenger arrival time distribution

In [2]:
import numpy as np
import random
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d


def energy_price_per_kWh(t, base=0.1363, m_peak=1.8, m_shoulder=1.05, m_off=0.7):
    """
    Time-of-Use (TOU) pricing function.
    t : float
        Hour of the day [0, 24).
    Returns
    -------
    float
        Electricity price in $/kWh.
    """
    if not (0 <= t < 24):
        raise ValueError("t must be in [0,24)")
    
    # Peak period: 4–9 pm
    if 16 <= t < 21:
        return base * m_peak
    # Off-peak: midnight–6 am, 9 pm–midnight
    elif 0 <= t < 6 or 21 <= t < 24:
        return base * m_off
    # Shoulder period: rest of the day
    else:
        return base * m_shoulder


def simulate_evtol_and_passengers(seed=60, plot=True):
    """
    Simulates passenger arrivals & eVTOL arrivals over a 24-hour period.
    Assigns charging needs for eVTOLs, applies TOU pricing, and plots arrivals.
    """
    np.random.seed(seed)

    # ---------------------------------------------------------
    # 1. Generate arrival times
    # ---------------------------------------------------------
    # Passengers (850 total, clustered demand)
    passenger_arrivals = np.concatenate([
        np.random.normal(400, 85, 320),
        np.random.normal(1050, 95, 310),
        np.random.normal(800, 250, 220)
    ])

    # eVTOLs (830 total, multiple peaks)
    evtol_arrivals = np.concatenate([
        np.random.normal(400, 90, 310),
        np.random.normal(600, 70, 120),
        np.random.normal(800, 80, 120),
        np.random.normal(1000, 60, 220),
        np.random.normal(1200, 70, 60)
    ])

    # Clamp to [0,1440] (24h in minutes)
    passenger_arrivals = np.clip(passenger_arrivals, 0, 1440)
    evtol_arrivals = np.clip(evtol_arrivals, 0, 1440)

    # ---------------------------------------------------------
    # 2. Assign SoC (random between 20%–60%)
    # ---------------------------------------------------------
    SoC = {j: random.uniform(20, 60) for j in range(len(evtol_arrivals))}

    # ---------------------------------------------------------
    # 3. Charging parameters
    # ---------------------------------------------------------
    SoC_min = 70.0   # required SoC (%)
    c = 150.0        # battery capacity (kWh)
    qr = 125         # charging rate (kW)
    charging_cost, p, finish_charging_period = {}, {}, {}

    # ---------------------------------------------------------
    # 4. Charging simulation with TOU pricing
    # ---------------------------------------------------------
    for j in range(len(evtol_arrivals)):
        arrival_time_min = evtol_arrivals[j]   # in minutes
        arrival_hour = arrival_time_min / 60.0  # convert to hours (0–24)

        if SoC[j] < SoC_min:  
            # Energy needed to reach SoC_min
            charge_needed = c * ((SoC_min - SoC[j]) / 100.0)

            # Charging duration in hours
            hours_to_charge = charge_needed / qr
            periods_to_charge = int(hours_to_charge * 6) + 1  # 10-min bins
            p[j] = periods_to_charge

            # Track cost
            total_cost = 0.0
            current_time_hr = arrival_hour
            remaining_periods = periods_to_charge

            while remaining_periods > 0 and current_time_hr < 24:
                # Electricity price at this hour
                price = energy_price_per_kWh(current_time_hr % 24)

                # Energy delivered this period (10-min = 1/6 hr)
                energy_this_period = (qr / 6)  # kWh
                total_cost += energy_this_period * price

                # Advance time by 10 min = 1/6 h
                current_time_hr += 1/6
                remaining_periods -= 1

            charging_cost[j] = total_cost
            finish_charging_period[j] = current_time_hr
        else:
            # Already above required SoC
            p[j] = 0
            finish_charging_period[j] = arrival_hour
            charging_cost[j] = 0

    # ---------------------------------------------------------
    # 5. Passenger destinations & fares
    # ---------------------------------------------------------
    num_destinations = 5
    destination_fares = {d: random.randint(160, 300) for d in range(1, num_destinations + 1)}
    destination_of_passenger = {k: random.randint(1, num_destinations) for k in range(len(passenger_arrivals))}
    fare_of_passenger = {k: destination_fares[destination_of_passenger[k]] for k in range(len(passenger_arrivals))}

    # ---------------------------------------------------------
    # 6. Plots (10-min bins, 24h)
    # ---------------------------------------------------------
    if plot:
        fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        # Passenger arrivals
        axs[0].hist(passenger_arrivals, bins=144, alpha=0.7, color='blue', edgecolor='black')
        axs[0].set_ylabel("Passenger Count")
        axs[0].set_title("Passenger arrivals (10-min bins, 24h)")
        axs[0].set_xlim(0, 1440)
        axs[0].set_xticks(np.arange(0, 1441, 120))  # every 2h
        axs[0].grid(True, alpha=0.3)

        # EVTOL arrivals
        axs[1].hist(evtol_arrivals, bins=144, alpha=0.7, color='green', edgecolor='black')
        axs[1].set_xlabel("Arrival Time (minutes)")
        axs[1].set_ylabel("EVTOL Count")
        axs[1].set_title("EVTOL arrivals (10-min bins, 24h)")
        axs[1].set_xlim(0, 1440)
        axs[1].set_xticks(np.arange(0, 1441, 120))  # every 2h
        axs[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    # ---------------------------------------------------------
    # 7. Return results
    # ---------------------------------------------------------
    return {
        "passenger_arrivals": passenger_arrivals,
        "evtol_arrivals": evtol_arrivals,
        "SoC": SoC,
        "charging_periods": p,
        "finish_charging": finish_charging_period,
        "charging_cost": charging_cost,
        "destinations": destination_of_passenger,
        "fares": fare_of_passenger
    }




## Joint Demand Scenario Generation with NHPP

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d


def generate_joint_demand_scenarios(passenger_arrivals, evtol_arrivals, n_scenarios=30, plot=True):
    """
    Generate joint demand scenarios (NHPP-based) for passenger and eVTOL arrivals.
    Works for any time horizon (e.g., 24h with 10-min bins = 144 periods).

    Parameters
    ----------
    passenger_arrivals : array-like
        Arrival times of passengers (minutes).
    evtol_arrivals : array-like
        Arrival times of eVTOLs (minutes).
    n_scenarios : int, optional
        Number of demand scenarios to generate (default=30).
    plot : bool, optional
        If True, plot sample scenarios and baseline rates.

    Returns
    -------
    scenarios : list of dict
        Each dict contains scenario-specific arrival rates and realizations:
        - id : scenario index
        - lambda_a : passenger rate vector (per time bin)
        - lambda_e : eVTOL rate vector (per time bin)
        - a_t : simulated passenger counts per time bin
        - e_t : simulated eVTOL counts per time bin
        - total_passengers : total passenger demand in this scenario
        - total_evtols : total eVTOL demand in this scenario
        - theoretical_corr : theoretical correlation baseline
        - empirical_corr : realized correlation in this scenario
    """

    # -----------------------------
    # 1. Define time horizon bins
    # -----------------------------
    horizon_minutes = 1440   # full 24h
    bin_size = 10            # 10-min resolution
    l_periods = horizon_minutes // bin_size
    bin_edges = np.linspace(0, horizon_minutes, l_periods + 1)
    time_periods = np.arange(1, l_periods + 1)

    # -----------------------------
    # 2. Histogram counts per bin
    # -----------------------------
    passenger_hist, _ = np.histogram(passenger_arrivals, bins=bin_edges)
    evtol_hist, _ = np.histogram(evtol_arrivals, bins=bin_edges)

    # -----------------------------
    # 3. Apply smoothing (Gaussian)
    # -----------------------------
    f_a = gaussian_filter1d(passenger_hist.astype(float), sigma=2)
    f_e = gaussian_filter1d(evtol_hist.astype(float), sigma=2)
    f_a, f_e = np.maximum(f_a, 0), np.maximum(f_e, 0)

    # -----------------------------
    # 4. Estimate residual variance
    # -----------------------------
    residuals_a = passenger_hist - f_a
    residuals_e = evtol_hist - f_e
    sigma_epsilon_sq = min(np.var(residuals_a), np.var(residuals_e))

    # -----------------------------
    # 5. Generate scenarios
    # -----------------------------
    scenarios = []
    for scenario_id in range(n_scenarios):
        epsilon_t = np.random.normal(0, np.sqrt(sigma_epsilon_sq), l_periods)
        
        # Add correlated noise to intensities
        lambda_a_omega = np.maximum(f_a + epsilon_t, 0)
        lambda_e_omega = np.maximum(f_e + epsilon_t, 0)

        # Realized arrivals (Poisson sampling)
        a_t_omega = np.random.poisson(lambda_a_omega)
        e_t_omega = np.random.poisson(lambda_e_omega)

        # Theoretical and empirical correlations
        numerator = sigma_epsilon_sq
        denominator = np.sqrt((f_a + sigma_epsilon_sq) * (f_e + sigma_epsilon_sq))
        theoretical_corr = np.mean(numerator / denominator)
        empirical_corr = np.corrcoef(a_t_omega, e_t_omega)[0, 1]

        scenarios.append({
            "id": scenario_id,
            "lambda_a": lambda_a_omega,
            "lambda_e": lambda_e_omega,
            "a_t": a_t_omega,
            "e_t": e_t_omega,
            "total_passengers": np.sum(a_t_omega),
            "total_evtols": np.sum(e_t_omega),
            "theoretical_corr": theoretical_corr,
            "empirical_corr": empirical_corr
        })

    # -----------------------------
    # 6. Plot sample scenarios
    # -----------------------------
    if plot:
        fig, axes = plt.subplots(3, 2, figsize=(15, 12))
        fig.suptitle("Joint Demand Scenarios (Sample of 6)", fontsize=16)

        for i, scenario in enumerate(scenarios[:6]):  # plot only first 6
            ax = axes[i // 2, i % 2]
            ax.plot(time_periods, scenario["a_t"], "o-", label="Passengers", color="blue")
            ax.plot(time_periods, scenario["e_t"], "s-", label="eVTOLs", color="green")
            ax.set_title(f"Scenario {scenario['id']+1}")
            ax.set_xlabel("Time Period (10-min bins)")
            ax.set_ylabel("Count")
            ax.legend()
            ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Plot baseline smoothed rates
        plt.figure(figsize=(12, 5))
        plt.plot(time_periods, f_a, "o-", label="Baseline Passenger Rate", color="blue")
        plt.plot(time_periods, f_e, "s-", label="Baseline eVTOL Rate", color="green")
        plt.title("Baseline Arrival Rates (Smoothed)")
        plt.xlabel("Time Period (10-min bins)")
        plt.ylabel("Rate")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return scenarios

## Scenario Reduction via Backward Selection


In [4]:
import numpy as np
import matplotlib.pyplot as plt


def scenario_reduction_backward_with_evtol_filter(scenarios, target_size=3, plot=True):
    """
    Reduce scenarios using backward selection while ensuring no scenarios
    without eVTOLs are selected.

    Parameters
    ----------
    scenarios : list
        List of scenario dictionaries from generate_joint_demand_scenarios().
    target_size : int, optional
        Desired number of reduced scenarios (default=2).
    plot : bool, optional
        If True, generate comparison and analysis plots.

    Returns
    -------
    list
        Reduced set of scenarios with updated probabilities.
    """
    # -----------------------------
    # 1. Filter out scenarios with no eVTOLs
    # -----------------------------
    scenarios_with_evtols = [s for s in scenarios if s['total_evtols'] > 0]

    # If not enough scenarios remain, just keep the top ones by eVTOL count
    if len(scenarios_with_evtols) < target_size:
        scenarios_sorted = sorted(scenarios, key=lambda x: x['total_evtols'], reverse=True)
        scenarios_with_evtols = scenarios_sorted[:target_size]

    # -----------------------------
    # 2. Initialize probabilities and feature vectors
    # -----------------------------
    n_original = len(scenarios_with_evtols)
    for scenario in scenarios_with_evtols:
        scenario['probability'] = 1.0 / n_original
        scenario['feature_vector'] = np.concatenate([scenario['a_t'], scenario['e_t']])

    working_scenarios = scenarios_with_evtols.copy()

    # -----------------------------
    # 3. Backward selection loop
    # -----------------------------
    while len(working_scenarios) > target_size:
        n_current = len(working_scenarios)
        distances = np.zeros((n_current, n_current))

        # Compute pairwise distances (Euclidean)
        for i in range(n_current):
            for j in range(i + 1, n_current):
                dist = np.linalg.norm(
                    working_scenarios[i]['feature_vector'] - working_scenarios[j]['feature_vector']
                )
                distances[i, j] = dist
                distances[j, i] = dist

        # Identify scenario to remove (min cost)
        min_cost, scenario_to_remove, nearest_neighbor = float('inf'), None, None
        for i in range(n_current):
            mask = np.ones(n_current, dtype=bool)
            mask[i] = False
            min_dist = np.min(distances[i, mask])
            cost = working_scenarios[i]['probability'] * min_dist

            if cost < min_cost:
                min_cost = cost
                scenario_to_remove = i
                nearest_neighbor = np.argmin([
                    distances[i, j] if j != i else float('inf') for j in range(n_current)
                ])

        # Transfer probability to nearest neighbor and remove scenario
        working_scenarios[nearest_neighbor]['probability'] += working_scenarios[scenario_to_remove]['probability']
        _ = working_scenarios.pop(scenario_to_remove)

    # -----------------------------
    # 4. Normalize probabilities
    # -----------------------------
    total_prob = sum(s['probability'] for s in working_scenarios)
    for s in working_scenarios:
        s['probability'] /= total_prob

    # -----------------------------
    # 5. Optional plots
    # -----------------------------
    if plot:
        l_periods = len(scenarios[0]['a_t'])
        time_periods = np.arange(1, l_periods + 1)
        colors = ['blue', 'green', 'red', 'purple', 'orange', 'brown']

        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        fig.suptitle('Reduced Scenarios Analysis', fontsize=16, fontweight='bold')

        # Passenger arrivals
        for i, s in enumerate(working_scenarios):
            axes[0, 0].plot(time_periods, s['a_t'], 'o-',
                            label=f"Scenario {s['id']+1} (p={s['probability']:.2f})",
                            color=colors[i % len(colors)], linewidth=2, markersize=6)
        axes[0, 0].set_title("Passenger Arrivals by Time Period")
        axes[0, 0].set_xlabel("Time Period (10-min intervals)")
        axes[0, 0].set_ylabel("Passenger Count")
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)

        # eVTOL arrivals
        for i, s in enumerate(working_scenarios):
            axes[0, 1].plot(time_periods, s['e_t'], 's-',
                            label=f"Scenario {s['id']+1} (p={s['probability']:.2f})",
                            color=colors[i % len(colors)], linewidth=2, markersize=6)
        axes[0, 1].set_title("eVTOL Arrivals by Time Period")
        axes[0, 1].set_xlabel("Time Period (10-min intervals)")
        axes[0, 1].set_ylabel("eVTOL Count")
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)

        # Scenario probabilities
        labels = [f"Scenario {s['id']+1}" for s in working_scenarios]
        probs = [s['probability'] for s in working_scenarios]
        bars = axes[1, 0].bar(labels, probs,
                              color=colors[:len(working_scenarios)], alpha=0.7)
        axes[1, 0].set_title("Scenario Probabilities")
        axes[1, 0].set_ylabel("Probability")
        axes[1, 0].grid(True, alpha=0.3)
        for bar, prob in zip(bars, probs):
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                            f"{prob:.2f}", ha="center", fontweight="bold")

        # Comparison of averages
        orig_p = [s['total_passengers'] for s in scenarios if s['total_evtols'] > 0]
        orig_e = [s['total_evtols'] for s in scenarios if s['total_evtols'] > 0]
        red_p = [s['total_passengers'] for s in working_scenarios]
        red_e = [s['total_evtols'] for s in working_scenarios]

        x = np.arange(2)
        w = 0.35
        axes[1, 1].bar(x - w/2, [np.mean(orig_p), np.mean(orig_e)], w,
                       label="Original Avg", alpha=0.6)
        axes[1, 1].bar(x + w/2, [np.mean(red_p), np.mean(red_e)], w,
                       label="Reduced Avg", alpha=0.8)
        axes[1, 1].set_xticks(x)
        axes[1, 1].set_xticklabels(["Passengers", "eVTOLs"])
        axes[1, 1].set_title("Original vs Reduced Scenarios")
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

    return working_scenarios


## Conditional Sampling of State of Charge (SoC)

In [5]:
import numpy as np
import matplotlib.pyplot as plt


def conditional_soc_sampling_per_evtol(
    reduced_scenarios,
    s_base=50,
    k=12,
    sigma_base=3,
    sigma_scale=5,
    plot=False
):
    """
    Perform conditional sampling of State of Charge (SoC) for each eVTOL 
    based on passenger-to-eVTOL load ratio at arrival times.
    
    SoC is constrained between 20 and 60.
    
    Parameters
    ----------
    reduced_scenarios : list
        Scenarios reduced from scenario_reduction_backward_with_evtol_filter().
    s_base : float, default=50
        Base SoC when load ratio is 0 (midpoint of 20-60 range).
    k : float, default=12
        Scaling factor for mean SoC decrease with load ratio.
    sigma_base : float, default=3
        Base standard deviation for SoC sampling.
    sigma_scale : float, default=5
        Scaling factor for std deviation increase with load ratio.
    plot : bool, default=False
        If True, will generate visualization of SoC distributions.
    
    Returns
    -------
    list
        The same reduced scenarios list, but with added SoC information:
        - scenario['rho_t_omega'] : load ratio per period
        - scenario['evtol_socs'] : dict of eVTOL ID → sampled SoC
        - scenario['evtol_arrival_periods'] : dict of eVTOL ID → arrival period
        - scenario['total_evtols_count'] : total number of eVTOLs in scenario
        - scenario['avg_load_ratio'] : average load ratio over periods
    """
    for scenario in reduced_scenarios:
        a_t_omega = scenario['a_t']  # Passenger arrivals per period
        e_t_omega = scenario['e_t']  # eVTOL arrivals per period

        # -----------------------------
        # Load ratio per period
        # -----------------------------
        rho_t_omega = np.zeros_like(a_t_omega, dtype=float)
        for t in range(len(a_t_omega)):
            if e_t_omega[t] > 0:
                rho_t_omega[t] = a_t_omega[t] / e_t_omega[t]
            else:
                # Handle zero eVTOL cases with interpolation
                if 0 < t < len(a_t_omega) - 1:
                    rho_t_omega[t] = (rho_t_omega[t-1] + rho_t_omega[t+1]) / 2
                elif t > 0:
                    rho_t_omega[t] = rho_t_omega[t-1]
                else:
                    rho_t_omega[t] = rho_t_omega[t+1] if len(a_t_omega) > 1 else 1.0

        evtol_socs = {}
        evtol_arrival_periods = {}
        evtol_count = 0

        # -----------------------------
        # Assign SoC per eVTOL
        # -----------------------------
        for t in range(len(e_t_omega)):
            for _ in range(e_t_omega[t]):
                evtol_id = evtol_count
                evtol_arrival_periods[evtol_id] = t + 1  # period index (1-based)

                load_ratio = rho_t_omega[t]
                mu_s = s_base - k * load_ratio
                sigma_s = sigma_base + sigma_scale * load_ratio

                # Clip mean and std dev
                mu_s = np.clip(mu_s, 25, 55)
                sigma_s = np.clip(sigma_s, 2, 8)

                # Sample until valid (within [20,60])
                sampled = False
                while not sampled:
                    soc_sample = np.random.normal(mu_s, sigma_s)
                    if 20 <= soc_sample <= 60:
                        evtol_socs[evtol_id] = soc_sample
                        sampled = True

                evtol_count += 1

        # -----------------------------
        # Store results
        # -----------------------------
        scenario['rho_t_omega'] = rho_t_omega
        scenario['evtol_socs'] = evtol_socs
        scenario['evtol_arrival_periods'] = evtol_arrival_periods
        scenario['total_evtols_count'] = evtol_count
        scenario['avg_load_ratio'] = np.mean(rho_t_omega)

    # -----------------------------
    # Optional Plot
    # -----------------------------
    if plot:
        plt.figure(figsize=(12, 6))
        for scenario in reduced_scenarios:
            socs = list(scenario['evtol_socs'].values())
            plt.hist(socs, bins=8, alpha=0.6, edgecolor="black",
                     label=f"Scenario {scenario['id']+1}")
        plt.title("SoC Distribution Across Reduced Scenarios")
        plt.xlabel("SoC (%)")
        plt.ylabel("Frequency")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

    return reduced_scenarios


## Input Data Preparation for Stochastic Optimization Model

In [6]:
import random
import numpy as np

def build_stochastic_model_input(
    reduced_scenarios,
    m_facilities=3,
    l_periods=144,        # full day in 10-min bins
    SoC_min=70.0,
    c=150.0,
    qr=125,
    LoS=2,
    eVTOL_capacity=4,
    destination_fares={1: 160, 2: 200, 3: 220, 4: 250, 5: 300},  # 5 destinations
    verbose=True
):
    """
    Build stochastic model input for optimization based on reduced scenarios.

    Parameters
    ----------
    reduced_scenarios : list
        Scenarios (after reduction + SoC sampling).
    m_facilities : int, default=3
        Number of facilities.
    l_periods : int, default=144
        Number of 10-min periods (24h horizon).
    SoC_min : float, default=70.0
        Minimum required state of charge (%).
    c : float, default=150.0
        Battery capacity (kWh).
    qr : float, default=125
        Charging rate (kW).
    LoS : int, default=1
        Level of service parameter.
    eVTOL_capacity : int, default=4
        Passenger capacity per eVTOL.
    destination_fares : dict
        Fare per passenger by destination.
    verbose : bool, default=True
        Print summaries.

    Returns
    -------
    dict
        Stochastic input dictionary with global parameters and scenario data.
    """
    stochastic_input = {
        'scenarios': {},
        'global_parameters': {
            'm_facilities': m_facilities,
            'l_periods': l_periods,
            'SoC_min': SoC_min,
            'c': c,
            'qr': qr,
            'LoS': LoS,
            'eVTOL_capacity': eVTOL_capacity,
            'destination_fares': destination_fares
        }
    }

    # Electricity price function (TOU pricing, could be extended)
    def price_per_kWh(t):
        hour = (t * 10) // 60  # convert period index to hour of day
        if 16 <= hour < 21:   # peak (4–9 pm)
            return 0.1363 * 1.35
        elif 0 <= hour < 6 or 21 <= hour < 24:  # off-peak
            return 0.1363 * 0.85
        else:  # shoulder
            return 0.1363 * 1.00

    # -----------------------------
    # Build scenario-specific data
    # -----------------------------
    for scenario in reduced_scenarios:
        sid = scenario['id']
        prob = scenario['probability']

        n_evtols = scenario['total_evtols_count']
        w_passengers = int(np.sum(scenario['a_t']))

        scenario_data = {
            'probability': prob,
            'n_evtols': n_evtols,
            'w_passengers': w_passengers,
            'evtols': {},
            'passengers': {},
            'w_matrix': {}
        }

        # ---- eVTOLs ----
        evtol_arrivals = []
        for t, count in enumerate(scenario['e_t']):
            for _ in range(int(count)):
                arrival_time = t * 10 + random.uniform(0, 10)
                evtol_arrivals.append(arrival_time)
        evtol_arrivals_sorted = sorted(evtol_arrivals)

        soc_values = [scenario['evtol_socs'][eid] for eid in sorted(scenario['evtol_socs'].keys())]

        for j, arrival_time in enumerate(evtol_arrivals_sorted):
            arrival_bin = int(arrival_time // 10) + 1
            soc = soc_values[j] if j < len(soc_values) else random.uniform(20, 60)

            if soc < SoC_min:
                charge_needed = c * ((SoC_min - soc) / 100.0)
                hours_to_charge = charge_needed / qr
                periods_to_charge = hours_to_charge * 6  # convert hours to 10-min periods
                p_j = int(periods_to_charge) + 1
                finish_charging_period = min(arrival_bin + p_j, l_periods)

                total_cost, current_period, remaining = 0, arrival_bin, p_j
                while remaining > 0 and current_period <= l_periods:
                    energy_this_period = min(1, remaining) * (qr / 6)
                    total_cost += energy_this_period * price_per_kWh(current_period)
                    remaining -= 1
                    current_period += 1
            else:
                p_j, finish_charging_period, total_cost = 0, arrival_bin, 0

            scenario_data['evtols'][j] = {
                'arrival_time': arrival_time,
                'arrival_bin': arrival_bin,
                'soc': soc,
                'charging_periods': p_j,
                'finish_charging_period': finish_charging_period,
                'charging_cost': total_cost
            }

        # ---- Passengers ----
        passenger_arrivals = []
        for t, count in enumerate(scenario['a_t']):
            for _ in range(int(count)):
                arrival_time = t * 10 + random.uniform(0, 10)
                passenger_arrivals.append(arrival_time)
        passenger_arrivals_sorted = sorted(passenger_arrivals)

        dest_of_passenger, fare_of_passenger = {}, {}
        for k, arrival_time in enumerate(passenger_arrivals_sorted):
            dest = random.randint(1, len(destination_fares))
            dest_of_passenger[k] = dest
            fare_of_passenger[k] = destination_fares[dest]
            arrival_bin = int(arrival_time // 10) + 1

            scenario_data['passengers'][k] = {
                'arrival_time': arrival_time,
                'arrival_bin': arrival_bin,
                'destination': dest,
                'fare': fare_of_passenger[k]
            }

        # ---- Similarity matrix ----
        w_matrix = {}
        for k in range(w_passengers):
            for q in range(w_passengers):
                if k != q and dest_of_passenger[k] == dest_of_passenger[q]:
                    w_matrix[(k, q)] = 1
                else:
                    w_matrix[(k, q)] = 0
        scenario_data['w_matrix'] = w_matrix

        stochastic_input['scenarios'][sid] = scenario_data

    # -----------------------------
    # Scenario ranges
    # -----------------------------
    all_evtols = [s['n_evtols'] for s in stochastic_input['scenarios'].values()]
    all_passengers = [s['w_passengers'] for s in stochastic_input['scenarios'].values()]
    stochastic_input['global_parameters']['n_evtols_range'] = (min(all_evtols), max(all_evtols))
    stochastic_input['global_parameters']['w_passengers_range'] = (min(all_passengers), max(all_passengers))

    # -----------------------------
    # Verbose summary
    # -----------------------------
    if verbose:
        print("="*80)
        print("STOCHASTIC MODEL INPUT SUMMARY")
        print("="*80)
        print(f"Number of scenarios: {len(stochastic_input['scenarios'])}\n")
        for sid, data in stochastic_input['scenarios'].items():
            print(f"Scenario {sid} (p={data['probability']:.3f}):")
            print(f"  - eVTOLs: {data['n_evtols']}")
            print(f"  - Passengers: {data['w_passengers']}")
            if data['evtols']:
                avg_soc = np.mean([e['soc'] for e in data['evtols'].values()])
                avg_cost = np.mean([e['charging_cost'] for e in data['evtols'].values()])
                print(f"  - Avg SoC: {avg_soc:.1f}% | Avg Charging Cost: ${avg_cost:.2f}")
            dests = [p['destination'] for p in data['passengers'].values()]
            counts = [dests.count(d) for d in set(dests)]
            print(f"  - Destinations: " + ", ".join([f"D{d}={counts[i]}" for i, d in enumerate(set(dests))]))
            print()
        print(f"Range of eVTOLs: {min(all_evtols)}–{max(all_evtols)}")
        print(f"Range of passengers: {min(all_passengers)}–{max(all_passengers)}")

    return stochastic_input


## Stochastic Optimization Model Fourmulation

In [7]:
# ========================================================
# Stochastic eVTOL scheduling model with Gurobi
# ========================================================
import gurobipy as grb


def build_and_solve_stochastic_model_gurobi(stochastic_input, M=1000, verbose=True):
    """
    Build and solve the stochastic eVTOL scheduling model using Gurobi.

    Parameters
    ----------
    stochastic_input : dict
        Input dictionary with global parameters and scenario data.
    M : int, default=1000
        Big-M constant for linking constraints.
    verbose : bool, default=True
        If False, suppress solver output.

    Returns
    -------
    model : gurobipy.Model
        The solved Gurobi model.
    results : dict
        Summary results (status and objective value).
    """

    # -----------------------------------------------------
    # Extract global parameters
    # -----------------------------------------------------
    global_params = stochastic_input['global_parameters']
    m_facilities = global_params['m_facilities']    # number of charging facilities
    l_periods = global_params['l_periods']          # number of discrete time periods
    eVTOL_capacity = global_params['eVTOL_capacity']
    LoS = global_params['LoS']                      # Level of Service (max wait time allowed)

    # Sets
    I = list(range(m_facilities))                   # facility indices
    Time = list(range(1, l_periods + 1))            # time periods (1..L)
    S = list(stochastic_input['scenarios'].keys())  # scenarios (s instead of Ω)

    # -----------------------------------------------------
    # Initialize Gurobi model
    # -----------------------------------------------------
    model = grb.Model("Stochastic_eVTOL_Scheduling_Model")
    if not verbose:
        model.setParam("OutputFlag", 0)

    # -----------------------------------------------------
    # Decision variables
    # -----------------------------------------------------
    # x[i,j,t,s] = 1 if eVTOL j assigned to facility i starting at time t in scenario s
    # y[j,k,s]   = 1 if passenger k assigned to eVTOL j in scenario s
    # C[j,s]     = completion time of charging for eVTOL j in scenario s
    # F[j,s]     = takeoff time of eVTOL j in scenario s
    # T[j,k,s]   = waiting time of passenger k assigned to eVTOL j in scenario s
    # discount_factor[j,k,s] = discount applied to passenger k in eVTOL j in scenario s
    x, y, C, F, Tvar, discount_factor = {}, {}, {}, {}, {}, {}

    for s in S:
        scenario = stochastic_input['scenarios'][s]
        n_evtols = scenario['n_evtols']
        w_passengers = scenario['w_passengers']

        # Facility assignment variables
        for i in I:
            for j in range(n_evtols):
                for t in Time:
                    x[i, j, t, s] = model.addVar(vtype=grb.GRB.BINARY,
                                                 name=f"x_{i}_{j}_{t}_{s}")

        # Passenger assignment variables
        for j in range(n_evtols):
            for k in range(w_passengers):
                y[j, k, s] = model.addVar(vtype=grb.GRB.BINARY,
                                          name=f"y_{j}_{k}_{s}")

        # Charging completion and takeoff times
        for j in range(n_evtols):
            C[j, s] = model.addVar(lb=0, vtype=grb.GRB.CONTINUOUS,
                                   name=f"C_{j}_{s}")
            F[j, s] = model.addVar(lb=0, vtype=grb.GRB.CONTINUOUS,
                                   name=f"F_{j}_{s}")

        # Passenger tardiness and discount
        for j in range(n_evtols):
            for k in range(w_passengers):
                Tvar[j, k, s] = model.addVar(lb=0, vtype=grb.GRB.CONTINUOUS,
                                             name=f"T_{j}_{k}_{s}")
                discount_factor[j, k, s] = model.addVar(lb=0, ub=1,
                                                        vtype=grb.GRB.CONTINUOUS,
                                                        name=f"discount_{j}_{k}_{s}")

    model.update()

    # -----------------------------------------------------
    # Constraints
    # -----------------------------------------------------
    for s in S:
        scenario = stochastic_input['scenarios'][s]
        n_evtols = scenario['n_evtols']
        w_passengers = scenario['w_passengers']
        J = list(range(n_evtols))     # eVTOLs in this scenario
        K = list(range(w_passengers)) # passengers in this scenario

        # Scenario-specific parameters
        r = {j: scenario['evtols'][j]['arrival_bin'] for j in J}
        p = {j: scenario['evtols'][j]['charging_periods'] for j in J}
        d = {k: scenario['passengers'][k]['arrival_bin'] for k in K}
        fare_of_passenger = {k: scenario['passengers'][k]['fare'] for k in K}
        charging_cost = {j: scenario['evtols'][j]['charging_cost'] for j in J}
        w = scenario['w_matrix']

        # (1) Charge Assignment
        for j in J:
            model.addConstr(
                grb.quicksum(x[i, j, t, s] for i in I
                             for t in range(r[j], l_periods - p[j] + 2)) >= 1,
                name=f"Charge_Assignment_{j}_{s}"
            )

        # (2) Facility Conflict
        for i in I:
            for t in Time:
                model.addConstr(
                    grb.quicksum(x[i, j, u, s]
                                 for j in J
                                 for u in range(max(1, t - p[j] + 1), t + 1)) <= 1,
                    name=f"Facility_Conflict_{i}_{t}_{s}"
                )

        # (3) Arrival Time
        for j in J:
            model.addConstr(
                grb.quicksum(x[i, j, t, s] for i in I for t in range(1, r[j])) == 0,
                name=f"Arrival_Time_{j}_{s}"
            )

        # (4) Passenger Service
        for k in K:
            model.addConstr(grb.quicksum(y[j, k, s] for j in J) <= 1,
                            name=f"Passenger_Service_{k}_{s}")

        # (5) eVTOL Capacity
        for j in J:
            model.addConstr(grb.quicksum(y[j, k, s] for k in K) <= eVTOL_capacity,
                            name=f"eVTOL_Capacity_{j}_{s}")

        # (6) Same Destination
        for j in J:
            for k in K:
                for q in K:
                    if q > k:
                        model.addConstr(y[j, k, s] + y[j, q, s] - 1 <= w[k, q],
                                        name=f"Same_Destination_{j}_{k}_{q}_{s}")

        # (7) Takeoff after charging
        for j in J:
            model.addConstr(F[j, s] >= C[j, s], name=f"Takeoff_After_Charge_{j}_{s}")

        # (8) Takeoff after passenger arrival
        for j in J:
            for k in K:
                model.addConstr(F[j, s] >= d[k] - M * (1 - y[j, k, s]),
                                name=f"Takeoff_After_Passenger_Arrival_{j}_{k}_{s}")

        # (9) Completion time definition
        for j in J:
            model.addConstr(C[j, s] >= grb.quicksum((t + p[j] - 1) * x[i, j, t, s]
                                                    for i in I
                                                    for t in range(1, l_periods - p[j] + 2)),
                            name=f"Completion_Time_Def_{j}_{s}")

        # (10) Tardiness definition + LoS
        for j in J:
            for k in K:
                model.addConstr(Tvar[j, k, s] >= F[j, s] - d[k] - M * (1 - y[j, k, s]))
                model.addConstr(Tvar[j, k, s] <= F[j, s] - d[k] + M * (1 - y[j, k, s]))
                model.addConstr(Tvar[j, k, s] >= 0)
                model.addConstr(Tvar[j, k, s] <= LoS * y[j, k, s],
                                name=f"LoS_Constraint_{j}_{k}_{s}")

        # (11) Discount function (piecewise linear)
        discount_breakpoints = [0, 1, 2]
        discount_values = [0, 0.05, 0.15]
        for j in J:
            for k in K:
                model.addGenConstrPWL(
                    Tvar[j, k, s], discount_factor[j, k, s],
                    discount_breakpoints, discount_values,
                    name=f"Discount_PWL_{j}_{k}_{s}"
                )
                model.addConstr(discount_factor[j, k, s] <= y[j, k, s],
                                name=f"Discount_Active_{j}_{k}_{s}")

    # -----------------------------------------------------
    # Objective Function
    # -----------------------------------------------------
    total_revenue, total_charging_cost, unserved_penalty = 0, 0, 0

    for s in S:
        scenario = stochastic_input['scenarios'][s]
        prob = scenario['probability']
        n_evtols = scenario['n_evtols']
        w_passengers = scenario['w_passengers']
        J = list(range(n_evtols))
        K = list(range(w_passengers))
        fare_of_passenger = {k: scenario['passengers'][k]['fare'] for k in K}
        charging_cost = {j: scenario['evtols'][j]['charging_cost'] for j in J}

        # Revenue
        scenario_revenue = grb.quicksum(
            y[j, k, s] * fare_of_passenger[k] * (1 - discount_factor[j, k, s])
            for j in J for k in K
        )
        total_revenue += prob * scenario_revenue

        # Charging cost
        scenario_charging_cost = grb.quicksum(
            x[i, j, t, s] * charging_cost[j]
            for i in I for j in J for t in Time
        )
        total_charging_cost += prob * scenario_charging_cost

        # Unserved penalty
        scenario_unserved_penalty = 10 * grb.quicksum(
            (1 - grb.quicksum(y[j, k, s] for j in J)) for k in K
        )
        unserved_penalty += prob * scenario_unserved_penalty

    model.setObjective(total_revenue - total_charging_cost - unserved_penalty,
                       grb.GRB.MAXIMIZE)

    # -----------------------------------------------------
    # Solve the Model
    # -----------------------------------------------------
    model.optimize()

    # -----------------------------------------------------
    # Check solution status
    # -----------------------------------------------------
    status = model.Status
    if status == grb.GRB.OPTIMAL:
        print(" Optimal solution found.")
        print(f"   Objective value = {model.ObjVal:.2f}")
        print(f"   Solutions stored = {model.SolCount}")
    elif status == grb.GRB.INFEASIBLE:
        print(" Model is infeasible. Writing IIS...")
        model.computeIIS()
        model.write("infeasible.ilp")
    elif status == grb.GRB.UNBOUNDED:
        print(" Model is unbounded.")
    elif status == grb.GRB.INF_OR_UNBD:
        print(" Model is infeasible or unbounded.")
    else:
        print(f"ℹ Optimization ended with status {status}")

    # -----------------------------------------------------
    # ✅ Detailed Reporting (only if feasible/optimal)
    # -----------------------------------------------------
    if status == grb.GRB.OPTIMAL:
        print(f"\n2. SUMMARY STATISTICS (EXPECTED VALUES):")
        expected_served_passengers = 0
        expected_unserved_passengers = 0
        expected_utilized_evtols = 0
        expected_revenue = 0
        expected_charging_cost = 0
        expected_unserved_penalty = 0
        expected_wait_times = []
        expected_discounts = []

        for s in S:
            scenario = stochastic_input['scenarios'][s]
            prob = scenario['probability']
            n_evtols = scenario['n_evtols']
            w_passengers = scenario['w_passengers']
            J = list(range(n_evtols))
            K = list(range(w_passengers))

            # Served/unserved stats
            served_passengers = sum(model.getVarByName(f"y_{j}_{k}_{s}").X > 0.5 for j in J for k in K)
            unserved_passengers = w_passengers - served_passengers
            utilized_evtols = sum(any(model.getVarByName(f"y_{j}_{k}_{s}").X > 0.5 for k in K) for j in J)

            # Revenue
            scenario_revenue = 0
            for j in J:
                for k in K:
                    if model.getVarByName(f"y_{j}_{k}_{s}").X > 0.5:
                        discount = model.getVarByName(f"discount_{j}_{k}_{s}").X
                        scenario_revenue += scenario['passengers'][k]['fare'] * (1 - discount)

            # Charging cost
            scenario_charging_cost = sum(
                model.getVarByName(f"x_{i}_{j}_{t}_{s}").X * scenario['evtols'][j]['charging_cost']
                for i in I for j in J for t in Time
            )
            scenario_unserved_penalty = 10 * unserved_passengers

            # Accumulate expectations
            expected_served_passengers += prob * served_passengers
            expected_unserved_passengers += prob * unserved_passengers
            expected_utilized_evtols += prob * utilized_evtols
            expected_revenue += prob * scenario_revenue
            expected_charging_cost += prob * scenario_charging_cost
            expected_unserved_penalty += prob * scenario_unserved_penalty

            # Wait times & discounts
            for j in J:
                for k in K:
                    if model.getVarByName(f"y_{j}_{k}_{s}").X > 0.5:
                        expected_wait_times.append(prob * model.getVarByName(f"T_{j}_{k}_{s}").X)
                        expected_discounts.append(prob * model.getVarByName(f"discount_{j}_{k}_{s}").X * 100)

            # Scenario-level summary
            print(f"\n   SCENARIO {s} DETAILS (p={prob:.2f}):")
            print(f"     Passengers: {w_passengers}, eVTOLs: {n_evtols}")
            print(f"     Served Passengers: {served_passengers}")
            print(f"     Unserved Passengers: {unserved_passengers}")
            print(f"     Utilized eVTOLs: {utilized_evtols}")
            print(f"     Revenue: ${scenario_revenue:.2f}")
            print(f"     Charging Cost: ${scenario_charging_cost:.2f}")
            print(f"     Unserved Penalty: ${scenario_unserved_penalty:.2f}")
            print(f"     Net Profit: ${scenario_revenue - scenario_charging_cost - scenario_unserved_penalty:.2f}")

        # Overall expected values
        print(f"\n   OVERALL EXPECTED VALUES:")
        print(f"     Expected Served Passengers: {expected_served_passengers:.2f}")
        print(f"     Expected Unserved Passengers: {expected_unserved_passengers:.2f}")
        print(f"     Expected Utilized eVTOLs: {expected_utilized_evtols:.2f}")
        print(f"     Expected Total Revenue: ${expected_revenue:.2f}")
        print(f"     Expected Total Charging Cost: ${expected_charging_cost:.2f}")
        print(f"     Expected Unserved Penalty: ${expected_unserved_penalty:.2f}")
        print(f"     Expected Net Profit: ${model.objVal:.2f}")

        if expected_wait_times:
            avg_wait = sum(expected_wait_times) / len(expected_wait_times)
            max_wait = max(expected_wait_times)
            print(f"     Expected Average Waiting Time: {avg_wait:.2f} periods")
            print(f"     Expected Maximum Waiting Time: {max_wait:.2f} periods")

        if expected_discounts:
            avg_discount = sum(expected_discounts) / len(expected_discounts)
            print(f"     Expected Average Discount: {avg_discount:.2f}%")

        print("\n" + "="*80)
        print("END OF STOCHASTIC RESULTS")
        print("="*80)

    # -----------------------------------------------------
    # Return model + results
    # -----------------------------------------------------
    results = {
        "status": model.Status,
        "objective": model.objVal if model.SolCount > 0 else None
    }

    return model, results


# Main Code

In [ ]:
if __name__ == "__main__":
    # Step 1: Generate arrivals from your simulation
#    from simulation import simulate_evtol_and_passengers  # your base simulation
    
    results = simulate_evtol_and_passengers(plot=False)

    # Step 2: Generate 30 scenarios (joint demand)
    scenarios = generate_joint_demand_scenarios(
        results["passenger_arrivals"],
        results["evtol_arrivals"],
        n_scenarios=35,
        plot=False  # skip plotting 30 at once
    )
    print(f"Generated {len(scenarios)} scenarios.")

    # Step 3: Reduce scenarios to 3 using backward reduction
    reduced_scenarios = scenario_reduction_backward_with_evtol_filter(
        scenarios,
        target_size=3,   
        plot=False        
    )
    # Step 4: Assign SoCs to each eVTOL in reduced scenarios with Conditional Sampling  
    scenarios_with_soc = conditional_soc_sampling_per_evtol(reduced_scenarios, plot=False)
    
    # -----------------------------
    # Step 5: Prepare stochastic input
    # -----------------------------
    stochastic_input = build_stochastic_model_input(
        scenarios_with_soc,
        m_facilities=3,
        l_periods=144,      
        SoC_min=70.0,
        verbose=True
    )


    # -----------------------------------------------------
    # Step 6: Build & Solve the stochastic model
    # -----------------------------------------------------
    print("\n=== Solving Stochastic Model ===")
    model, results = build_and_solve_stochastic_model_gurobi(
        stochastic_input,
        M=1000,
        verbose=True
    )


Generated 35 scenarios.
STOCHASTIC MODEL INPUT SUMMARY
Number of scenarios: 3

Scenario 14 (p=0.286):
  - eVTOLs: 828
  - Passengers: 863
  - Avg SoC: 38.5% | Avg Charging Cost: $8.50
  - Destinations: D1=174, D2=173, D3=156, D4=180, D5=180

Scenario 29 (p=0.143):
  - eVTOLs: 848
  - Passengers: 882
  - Avg SoC: 38.8% | Avg Charging Cost: $8.52
  - Destinations: D1=165, D2=178, D3=180, D4=182, D5=177

Scenario 32 (p=0.571):
  - eVTOLs: 868
  - Passengers: 906
  - Avg SoC: 39.1% | Avg Charging Cost: $8.24
  - Destinations: D1=202, D2=165, D3=181, D4=178, D5=180

Range of eVTOLs: 828–868
Range of passengers: 863–906


'\n\n\n    # -----------------------------------------------------\n    # Step 6: Build & Solve the stochastic model\n    # -----------------------------------------------------\n    print("\n=== Solving Stochastic Model ===")\n    model, results = build_and_solve_stochastic_model_gurobi(\n        stochastic_input,\n        M=1000,\n        verbose=True\n    )\n'